# Road PCI Intelligence — Evaluation: Extended Threshold Sweep & Non-Empty Mask Audit

This notebook evaluates the trained **Run A (Masked Baseline)** checkpoint (`best.pt`) on the **validation split only** without any retraining:
1. Decision threshold sweep across $\{0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9\}$ for all three sources (`crack500`, `road_crack`, `pothole`).
2. Exact empty ground-truth vs non-empty ground-truth metric breakdown, specifically auditing the number of `road_crack` validation images with $\ge 1$ crack pixel and their mean Dice score.


In [ ]:
# ==============================================================================
# 1. SETUP ENVIRONMENT & DEPENDENCIES
# ==============================================================================
import os
import sys
import subprocess
from pathlib import Path

print("Installing segmentation-models-pytorch and albumentations...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch==0.5.0", "albumentations==2.0.8"], check=True)

# Clone latest repo
repo_dir = Path('/kaggle/working/repo')
if not repo_dir.exists():
    print("Cloning GitHub repository...")
    subprocess.run(["git", "clone", "https://github.com/skilloutofbounds/road-pci-intelligence.git", str(repo_dir)], check=True)
else:
    print("Pulling latest GitHub changes...")
    subprocess.run(["git", "-C", str(repo_dir), "pull"], check=True)

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

import torch
import albumentations as A
import segmentation_models_pytorch as smp

from src.train import seed_everything, log_library_versions

seed_everything(42)
log_library_versions()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Active Device: {device}")


In [ ]:
# ==============================================================================
# 2. RESOLVE DATASET ROOTS & MANIFEST
# ==============================================================================
import pandas as pd
from src.build_manifest import DATASET_SPECS, resolve_dataset_root
from src.dataset import RoadPciDataset, build_eval_dataloader

manifest_path = repo_dir / 'manifests/segmentation_manifest.csv'
assert manifest_path.exists(), f"Missing manifest: {manifest_path}"
manifest_df = pd.read_csv(manifest_path)
print(f"Loaded segmentation manifest: {len(manifest_df):,} records")

dataset_roots = {}
for src in ['pothole', 'road_crack', 'crack500']:
    spec = DATASET_SPECS[src]
    root = resolve_dataset_root(spec['owner'], spec['name'])
    assert root is not None and root.exists(), f"Could not find dataset root for {src}"
    dataset_roots[src] = root
    print(f"  • {src:10s} root: {root}")

val_loader = build_eval_dataloader(
    manifest_df=manifest_df,
    dataset_roots=dataset_roots,
    split='val',
    batch_size=1,
    num_workers=2
)
print(f"Built validation DataLoader: {len(val_loader)} images")


In [ ]:
# ==============================================================================
# 3. LOCATE AND LOAD RUN A BEST CHECKPOINT
# ==============================================================================
k_input = Path('/kaggle/input')
candidates = list(k_input.rglob('best.pt'))
print(f"Discovered checkpoints in /kaggle/input: {candidates}")
assert len(candidates) > 0, "CRITICAL: No best.pt found in /kaggle/input!"

best_pt_path = candidates[0]
print(f"Using checkpoint: {best_pt_path}")

model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights=None,
    in_channels=3,
    classes=2
)
state_dict = torch.load(best_pt_path, map_location=device)
model.load_state_dict(state_dict)
model.to(device)
model.eval()
print("[OK] Loaded model weights successfully.")


In [ ]:
# ==============================================================================
# 4. RUN EXTENDED THRESHOLD SWEEP {0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9}
# ==============================================================================
from src.train import run_threshold_sweep, print_threshold_table
import json

extended_thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
print("Running extended threshold sweep on validation split...")
sweep_results = run_threshold_sweep(
    model=model,
    dataloader=val_loader,
    device=device,
    thresholds=extended_thresholds
)

print_threshold_table(sweep_results)

# Save sweep results to json
out_sweep_file = Path('/kaggle/working/run_a_extended_threshold_sweep.json')
with open(out_sweep_file, 'w', encoding='utf-8') as f:
    json.dump(sweep_results, f, indent=2)
print(f"[OK] Saved sweep results to {out_sweep_file}")


In [ ]:
# ==============================================================================
# 5. AUDIT ROAD_CRACK VALIDATION NON-EMPTY MASKS
# ==============================================================================
import numpy as np

# Find results at threshold 0.5
th_05_res = next(r for r in sweep_results if abs(r['threshold'] - 0.5) < 1e-5)

print("=" * 80)
print("ROAD CRACK VALIDATION NON-EMPTY MASK AUDIT (Threshold 0.5)")
print("=" * 80)
n_total = th_05_res['road_crack_images']
n_non_empty = th_05_res['road_crack_non_empty_images']
n_empty = n_total - n_non_empty
mean_img_dice_all = th_05_res['road_crack_mean_image_dice']
mean_img_dice_non_empty = th_05_res['road_crack_non_empty_mean_image_dice']

print(f"Total road_crack val images: {n_total}")
print(f"Images with >= 1 crack pixel (non-empty): {n_non_empty}")
print(f"Images with 0 crack pixels (all background): {n_empty}")
print(f"Mean Img Dice (over all {n_total} images): {mean_img_dice_all:.4f}")
print(f"Mean Img Dice (over only {n_non_empty} non-empty images): {mean_img_dice_non_empty:.4f}")
print("=" * 80)
